In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score


n = 3000

df = pd.DataFrame({
    "lead_score": np.random.normal(50, 12, n),
    "income": np.random.normal(65000, 18000, n),
    "days_to_first_contact": np.random.randint(0, 15, n),
    "num_calls_first_week": np.random.poisson(3, n),
})

#Create a true probability of sale from legitimate features
logit = (
    -2.0
    + 0.05 * df["lead_score"]
    + 0.00002 * df["income"]
    - 0.18 * df["days_to_first_contact"]
    + 0.22 * df["num_calls_first_week"]
)

p = 1 / (1 + np.exp(-logit))
df["sold"] = np.random.binomial(1, p, n)
df.head()

,lead_score,income,days_to_first_contact,num_calls_first_week,sold
0,41.809609,78542.289815,13,6,0
1,42.271303,60023.457739,13,2,0
2,49.621580,67656.842495,2,4,1
3,44.420344,66277.011933,11,3,1
4,52.730708,81620.328531,3,1,1


In [3]:
#Add an intentionally leaked feature
#"final_discount" is only known after the sale process finishes.
#We make it strongly related to sold, which will create fake-great performance.
df["final_discount"] = np.where(
    df["sold"] == 1,
    np.random.normal(3500, 700, n),   #sold cars tend to have meaningful final discounts
    np.random.normal(300, 200, n)     #unsold leads don't really get final discounts
)

#Keep discounts non-negative
df["final_discount"] = np.clip(df["final_discount"], 0, None)

In [4]:
#Compare a legit model vs leaked model

legit_features = [
    "lead_score",
    "income",
    "days_to_first_contact",
    "num_calls_first_week"
]

leaked_features = legit_features + ["final_discount"]

X_legit = df[legit_features]
X_leaked = df[leaked_features]
y = df["sold"]

X_train_legit, X_test_legit, y_train, y_test = train_test_split(
    X_legit, y, test_size=0.3, random_state=42, stratify=y
)

X_train_leaked, X_test_leaked, _, _ = train_test_split(
    X_leaked, y, test_size=0.3, random_state=42, stratify=y
)

#Legit model
model_legit = LogisticRegression(max_iter=2000)
model_legit.fit(X_train_legit, y_train)
pred_legit = model_legit.predict(X_test_legit)
proba_legit = model_legit.predict_proba(X_test_legit)[:, 1]

#Leaked model
model_leaked = LogisticRegression(max_iter=2000)
model_leaked.fit(X_train_leaked, y_train)
pred_leaked = model_leaked.predict(X_test_leaked)
proba_leaked = model_leaked.predict_proba(X_test_leaked)[:, 1]

print(f"Legit model accuracy: {accuracy_score(y_test, pred_legit):.3f}")
print(f"Legit model AUC:      {roc_auc_score(y_test, proba_legit):.3f}")
print()
print(f"Leaked model accuracy: {accuracy_score(y_test, pred_leaked):.3f}")
print(f"Leaked model AUC:      {roc_auc_score(y_test, proba_leaked):.3f}")

Legit model accuracy: 0.767
Legit model AUC:      0.758

Leaked model accuracy: 0.999
Leaked model AUC:      1.000


In [5]:
#Preprocessing leakage demo
#scaling the entire dataset BEFORE the split.

X = df[legit_features].copy()

#WRONG: scale on full data, then split
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)

X_train_wrong, X_test_wrong, y_train_wrong, y_test_wrong = train_test_split(
    X_scaled_wrong, y, test_size=0.3, random_state=42, stratify=y
)

model_wrong = LogisticRegression(max_iter=2000)
model_wrong.fit(X_train_wrong, y_train_wrong)
proba_wrong = model_wrong.predict_proba(X_test_wrong)[:, 1]
pred_wrong = model_wrong.predict(X_test_wrong)

#CORRECT: split first, then fit scaler only on training data
X_train_right, X_test_right, y_train_right, y_test_right = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler_right = StandardScaler()
X_train_right_scaled = scaler_right.fit_transform(X_train_right)
X_test_right_scaled = scaler_right.transform(X_test_right)

model_right = LogisticRegression(max_iter=2000)
model_right.fit(X_train_right_scaled, y_train_right)
proba_right = model_right.predict_proba(X_test_right_scaled)[:, 1]
pred_right = model_right.predict(X_test_right_scaled)

print("Wrong approach: scale full data before split")
print(f"Accuracy: {accuracy_score(y_test_wrong, pred_wrong):.3f}")
print(f"AUC:      {roc_auc_score(y_test_wrong, proba_wrong):.3f}")

print("\nCorrect approach: split first, then fit scaler on training data only")
print(f"Accuracy: {accuracy_score(y_test_right, pred_right):.3f}")
print(f"AUC:      {roc_auc_score(y_test_right, proba_right):.3f}")

Wrong approach: scale full data before split
Accuracy: 0.768
AUC:      0.758

Correct approach: split first, then fit scaler on training data only
Accuracy: 0.768
AUC:      0.758


In [6]:
# -----------------------------
# 5. Show the train-only vs full-data scaling parameters
# -----------------------------
full_means = pd.Series(scaler_wrong.mean_, index=legit_features, name="Full Data Mean")
train_means = pd.Series(scaler_right.mean_, index=legit_features, name="Train Only Mean")

comparison = pd.concat([full_means, train_means], axis=1)
comparison["Difference"] = comparison["Full Data Mean"] - comparison["Train Only Mean"]

print("\n=== WHY PREPROCESSING BEFORE SPLIT IS LEAKAGE ===")
print(comparison.round(3))


=== WHY PREPROCESSING BEFORE SPLIT IS LEAKAGE ===
                       Full Data Mean  Train Only Mean  Difference
lead_score                     49.968           49.669       0.299
income                      64640.530        64734.154     -93.623
days_to_first_contact           6.919            6.874       0.046
num_calls_first_week            3.009            3.013      -0.004
